# Etapa 2 — Coleta de dados

Baixamos cada série **na frequência original** e salvamos **sem tratamento** em `dados/brutos/`.
Transformar frequência, cortar datas e calcular log-diferenças fica para a etapa 3.

| Bloco | Variável | Fonte | Código |
|---|---|---|---|
| Dependente | PTAX de venda (R$/US$) | SGS/BCB | 1 |
| Externa | Treasury 10 anos (% a.a.) | FRED | DGS10 |
| Externa | VIX (pontos) | FRED | VIXCLS |
| Externa | Dólar vs. economias avançadas (índice) | FRED | DTWEXAFEGS |
| Doméstica | Meta Selic (% a.a.) | SGS/BCB | 432 |
| Doméstica | CDS 5 anos Brasil (pontos-base) | Investing.com (CSV manual) | — |
| Doméstica | Expectativa de resultado primário (% PIB) | Focus/BCB | — |
| Checagem | EMBI+ Brasil (pontos-base, até jul/2024) | IPEADATA | JPM366_EMBI366 |

In [1]:
import os
from pathlib import Path

import pandas as pd
import requests
from bcb import sgs, Expectativas
from fredapi import Fred

INICIO = "2006-01-01"                   # limitado pelo início dos índices de dólar do FRED
HOJE = pd.Timestamp.today().normalize()  # usado para cortar datas futuras
PASTA = Path("dados/brutos")
PASTA.mkdir(parents=True, exist_ok=True)

## 1. SGS — PTAX e meta Selic

A API do SGS limita séries **diárias** a 10 anos por consulta. Por isso a função abaixo baixa em
pedaços de 5 anos e junta tudo com `pd.concat`.

Lembre da etapa 1: a série 432 vem preenchida **até a próxima reunião do Copom** (datas futuras).
Cortamos tudo depois de hoje já aqui, porque um dado que ainda não aconteceu não é "bruto", é um erro.

In [2]:
def baixar_sgs(codigo, nome, inicio=INICIO, fim=HOJE, anos_por_bloco=5):
    """Baixa uma série diária do SGS em blocos para respeitar o limite de 10 anos da API."""
    blocos = []
    ini = pd.Timestamp(inicio)
    while ini <= fim:
        fim_bloco = min(ini + pd.DateOffset(years=anos_por_bloco) - pd.Timedelta(days=1), fim)
        blocos.append(sgs.get({nome: codigo}, start=ini, end=fim_bloco))
        ini = fim_bloco + pd.Timedelta(days=1)
    serie = pd.concat(blocos)
    serie = serie[~serie.index.duplicated()]   # evita datas repetidas na fronteira dos blocos
    return serie[serie.index <= HOJE]           # corta datas futuras

ptax = baixar_sgs(1, "ptax_venda")
selic = baixar_sgs(432, "selic_meta")

ptax.to_csv(PASTA / "ptax_venda.csv")
selic.to_csv(PASTA / "selic_meta.csv")
print(ptax.tail(3), "\n")
print(selic.tail(3))

            ptax_venda
Date                  
2026-11-04      5.4778
2026-11-05      5.4621
2026-11-06      5.5013 

            selic_meta
Date                  
2026-11-05       13.75
2026-11-06       13.75
2026-11-07       13.75


## 2. FRED — Treasury 10a, VIX e dólar vs. economias avançadas

Os `NaN` que aparecem são feriados americanos (o mercado não abre). **Não** preenchemos nada aqui;
isso é tratamento (etapa 3).

In [3]:
fred = Fred(api_key=os.environ["FRED_API_KEY"])

series_fred = {"ust10": "DGS10", "vix": "VIXCLS", "dolar_afe": "DTWEXAFEGS"}
externas = pd.DataFrame(
    {nome: fred.get_series(codigo, observation_start=INICIO) for nome, codigo in series_fred.items()}
)
externas.index.name = "Date"

externas.to_csv(PASTA / "externas_fred.csv")
externas.tail()

,ust10,vix,dolar_afe
Date,,,
2026-10-30,4.94,19.42,114.3735
2026-11-02,4.91,21.15,NaN
2026-11-03,4.95,20.83,NaN
2026-11-04,4.90,16.61,NaN
2026-11-05,4.84,28.18,NaN


## 3. CDS 5 anos — CSV do Investing.com

O arquivo vem no formato americano: datas `MM/DD/AAAA` e ordem decrescente. Ficamos só com a coluna
`Price` (fechamento), convertemos as datas e ordenamos do mais antigo para o mais recente.

**Teste de qualidade:** séries de mercado pouco líquidas às vezes repetem o último preço quando não há
negócio ("preço velho", *stale price*). Se um ano tem muitos dias com variação zero, desconfiamos dele.

In [4]:
arquivo_cds = PASTA / "cds_investing.csv"

cds = pd.read_csv(arquivo_cds, usecols=["Date", "Price"], thousands=",")
cds["Date"] = pd.to_datetime(cds["Date"], format="%m/%d/%Y")
cds = cds.rename(columns={"Price": "cds_5a"}).set_index("Date").sort_index()

# Fração de dias com variação exatamente zero, por ano
dias_parados = (cds["cds_5a"].diff() == 0).groupby(cds.index.year).mean()
print("Fração de dias sem variação, por ano:")
print(dias_parados.round(2).to_string())

cds.to_csv(PASTA / "cds_5a.csv")

Fração de dias sem variação, por ano:
Date
2007    0.89
2008    0.62
2009    0.02
2010    0.03
2011    0.01
2012    0.02
2013    0.01
2014    0.03
2015    0.00
2016    0.01
2017    0.02
2018    0.03
2019    0.04
2020    0.03
2021    0.01
2022    0.02
2023    0.01
2024    0.06
2025    0.01
2026    0.03


## 4. EMBI+ Brasil — IPEADATA (série de checagem)

Inativa desde jul/2024, mas serve para conferir se o CDS conta a mesma história no período comum.

In [5]:
url = "http://www.ipeadata.gov.br/api/odata4/ValoresSerie(SERCODIGO='JPM366_EMBI366')"
valores = requests.get(url, timeout=120).json()["value"]

embi = pd.DataFrame(valores)[["VALDATA", "VALVALOR"]]
embi["Date"] = pd.to_datetime(embi["VALDATA"].str[:10])   # a data vem com fuso horário; ficamos com AAAA-MM-DD
embi = embi.rename(columns={"VALVALOR": "embi"}).set_index("Date")[["embi"]]
embi = embi[embi.index >= INICIO]

embi.to_csv(PASTA / "embi_ipea.csv")
embi.tail(3)

,embi
Date,
2024-07-26,224.0
2024-07-29,225.0
2024-07-30,228.0


## 5. Focus — expectativa de resultado primário (% do PIB)

Cada linha do Focus é: *na data `Data`, a mediana das projeções para o ano `DataReferencia` era `Mediana`*.
Para cada semana há projeções para vários anos (corrente, seguinte...). Baixamos todos; na etapa 3
combinamos ano corrente e seguinte num horizonte fixo de 12 meses.

`baseCalculo == 0` = mediana com projeções dos últimos 30 dias (série padrão do Focus).
Sem esse filtro, cada data apareceria duplicada.

Convenção de sinal: **positivo = superávit**. Piora fiscal esperada = número caindo.

In [6]:
ep = Expectativas().get_endpoint("ExpectativasMercadoAnuais")

focus = (
    ep.query()
    .filter(ep.Indicador == "Resultado primário", ep.baseCalculo == 0)
    .filter(ep.Data >= "2005-01-01")    # um ano antes do INICIO, para o horizonte de 12 meses
    .select(ep.Data, ep.DataReferencia, ep.Mediana, ep.numeroRespondentes)
    .collect()
)
focus["Data"] = pd.to_datetime(focus["Data"])
focus = focus.sort_values(["Data", "DataReferencia"])

focus.to_csv(PASTA / "focus_primario.csv", index=False)
focus.tail(6)

,Data,DataReferencia,Mediana,numeroRespondentes
38403,2026-10-30,2030,-0.0125,46.0
38404,2026-10-30,2031,0.2000,43.0
38405,2026-10-30,2032,0.3752,42.0
38406,2026-10-30,2033,0.6086,42.0
38407,2026-10-30,2034,0.8000,42.0
38408,2026-10-30,2035,0.8000,41.0


## 6. Resumo da coleta

Uma linha por série: quantas observações, de quando a quando e quantos valores faltantes.

In [7]:
def resumo(df, col):
    s = df[col]
    return {"série": col, "n_obs": s.notna().sum(), "início": s.first_valid_index().date(),
            "fim": s.last_valid_index().date(), "NaN": s.isna().sum()}

linhas = [resumo(ptax, "ptax_venda"), resumo(selic, "selic_meta"), resumo(cds, "cds_5a"), resumo(embi, "embi")]
linhas += [resumo(externas, c) for c in externas.columns]
linhas.append({"série": "focus_primario", "n_obs": len(focus), "início": focus["Data"].min().date(),
               "fim": focus["Data"].max().date(), "NaN": focus["Mediana"].isna().sum()})
pd.DataFrame(linhas).set_index("série")

,n_obs,início,fim,NaN
série,,,,
ptax_venda,5234,2006-01-02,2026-11-06,0
selic_meta,7616,2006-01-01,2026-11-07,0
cds_5a,4917,2007-12-19,2026-11-06,0
embi,4783,2006-01-02,2024-07-30,1078
ust10,5216,2006-01-03,2026-11-05,223
vix,5277,2006-01-03,2026-11-05,162
dolar_afe,5222,2006-01-02,2026-10-30,217
focus_primario,38409,2005-01-03,2026-10-30,0
